In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import re

In [2]:
df = pd.read_csv(r'D:\nlp-basics\Text Representaion\IMDB Dataset.csv')

In [57]:
df.drop_duplicates(inplace=True)

## data cleaning

* it contains html tags
* there are words like this: Halliwell\'s :we need to convert it into like: Halliwell's :this
* lowercaing
* check for emojis then remove
* removing urls check first
* stopwords
* tokenization
* stemming

In [3]:
def remove_html_tags(text):
    if isinstance(text,str):
        soup = BeautifulSoup(text,'html.parser')
        clean_text = soup.get_text(separator=' ',strip=True)
    else:
        pass
    return clean_text

df['review'] = df['review'].apply(remove_html_tags)

C:\Users\bhosa\AppData\Local\Temp\ipykernel_13448\2213511721.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text,'html.parser')


In [4]:
def remove_puncuations(text):
    puncs = '"#$%&()*+,-./:;<=>?@[\\]^_`{|}~'
    return text.translate(str.maketrans("","",puncs))

df['review'] = df['review'].apply(remove_puncuations)

In [5]:
import emoji
def convert_emoji_text(text):
    clean_text = emoji.demojize(text)
    clean_text = clean_text.replace(":","")
    clean_text = clean_text.replace("_","")
    return clean_text

df['review'] = df['review'].apply(convert_emoji_text)

## corpus

In [125]:
total_words = []
for sent in df['review'].tolist():
    words =  sent.split(' ')
    total_words.extend(words)

In [138]:
unique_words = set(total_words)
print(f'total words in df[review] {len(unique_words)}')

total words in df[review] 225739


## BOW

In [6]:
import spacy
from nltk.stem.porter import PorterStemmer

nlp  = spacy.load('en_core_web_sm',disable=['parser','tagger','ner'])
ps = PorterStemmer()

def spacy_tokenizer(data_list):
    updated_text = []
    for doc in nlp.pipe(data_list,batch_size=1000,n_process=2):       
        tokens = [ps.stem(token.text)  if token.is_alpha else token.text for token in doc]
        updated_text.append(" ".join(tokens))
    return updated_text

In [7]:
temp = df.head(5)
raj = spacy_tokenizer(temp['review'].tolist())

In [8]:
%%time
updated_text = spacy_tokenizer(df['review'].tolist())

CPU times: total: 5min 27s
Wall time: 12min 48s


In [9]:
df['stem_tokenized_text'] = updated_text

In [32]:
df.head()

,review,sentiment,stem_tokenized_text
0,One of the other reviewers has mentioned that ...,positive,one of the other review ha mention that after ...
1,A wonderful little production The filming tech...,positive,a wonder littl product the film techniqu is ve...
2,I thought this was a wonderful way to spend ti...,positive,i thought thi wa a wonder way to spend time on...
3,Basically there's a family where a little boy ...,negative,basic there 's a famili where a littl boy jake...
4,Petter Mattei's Love in the Time of Money is a...,positive,petter mattei 's love in the time of money is ...


In [15]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
cv = CountVectorizer(stop_words ="english")

In [16]:
bow  = cv.fit_transform(df['stem_tokenized_text'])

In [26]:
import numpy as np
np.set_printoptions(threshold=np.inf)
bow[0].toarray().shape

(1, 127088)

In [31]:
cv.vocabulary_

{'review': 93388,
 'ha': 48915,
 'mention': 70674,
 'watch': 121584,
 'just': 59570,
 'oz': 82400,
 'episod': 36640,
 'll': 65397,
 'hook': 53133,
 'right': 93798,
 'thi': 112061,
 'exactli': 37678,
 'happen': 49861,
 'thing': 112120,
 'struck': 107509,
 'wa': 120858,
 'brutal': 17494,
 'unflinch': 117627,
 'scene': 97122,
 'violenc': 120283,
 'set': 99657,
 'word': 124489,
 'trust': 115561,
 'faint': 38858,
 'heart': 50740,
 'timid': 113319,
 'pull': 89601,
 'punch': 89666,
 'regard': 92273,
 'drug': 33575,
 'sex': 99816,
 'hardcor': 49980,
 'classic': 23023,
 'use': 118798,
 'nicknam': 77113,
 'given': 46069,
 'oswald': 81417,
 'maximum': 69454,
 'secur': 98422,
 'state': 106134,
 'penitentari': 84140,
 'focus': 42057,
 'mainli': 67650,
 'emerald': 35752,
 'citi': 22832,
 'experiment': 38142,
 'section': 98413,
 'prison': 88450,
 'cell': 20576,
 'glass': 46142,
 'face': 38667,
 'inward': 57054,
 'privaci': 88484,
 'high': 51819,
 'agenda': 5160,
 'em': 35682,
 'home': 52903,
 'manyar

## N- grams

In [140]:
from sklearn.feature_extraction.text import CountVectorizer
ngrams = CountVectorizer(ngram_range=(2,2))

In [161]:
grams = ngrams.fit_transform(df['stem_tokenized_text'])

In [162]:
ngrams.vocabulary_

{'one of': 1311005,
 'of the': 1288995,
 'the other': 1808164,
 'other review': 1333831,
 'review ha': 1515338,
 'ha mention': 792509,
 'mention that': 1146750,
 'that after': 1781972,
 'after watch': 55748,
 'watch just': 2000888,
 'just oz': 997798,
 'oz episod': 1350889,
 'episod you': 588979,
 'you ll': 2097054,
 'll be': 1073742,
 'be hook': 208241,
 'hook they': 868388,
 'they are': 1834754,
 'are right': 145054,
 'right as': 1521446,
 'as thi': 163297,
 'thi is': 1839969,
 'is exactli': 954469,
 'exactli what': 609427,
 'what happen': 2022171,
 'happen with': 805074,
 'with me': 2060835,
 'me the': 1136269,
 'the first': 1799923,
 'first thing': 672715,
 'thing that': 1845566,
 'that struck': 1789223,
 'struck me': 1721520,
 'me about': 1134698,
 'about oz': 28062,
 'oz wa': 1350965,
 'wa it': 1986898,
 'it brutal': 965419,
 'brutal and': 288735,
 'and unflinch': 116050,
 'unflinch scene': 1934428,
 'scene of': 1563536,
 'of violenc': 1290274,
 'violenc which': 1976802,
 'which 

In [163]:
grams[0].shape

(1, 2107509)

## TF-IDF

In [164]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [166]:
tf = tfidf.fit_transform(df['stem_tokenized_text'])

In [167]:
tf.shape

(49578, 127347)

In [168]:
tf[0].toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  